# Posterior Sampling (PS) Notebook

This notebook is a procedural (non-class) version for quick testing.

In [ ]:
import os
import sys
import math
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

from PIL import Image
from torchvision import transforms
from diffusers.utils.torch_utils import randn_tensor
from diffusers.models.embeddings import get_2d_rotary_pos_embed
from omegaconf import OmegaConf

ROOT = os.path.abspath("..")
if ROOT not in sys.path:
    sys.path.append(ROOT)

from pixelflow.utils import config as config_utils
from pixelflow.utils.misc import seed_everything
from pixelflow.scheduling_pixelflow import PixelFlowScheduler
from pixelflow.MCMCSampler import MCMCSampler
from inpaintingStart import get_operator

torch.set_grad_enabled(False)

In [ ]:
# ===== Config =====
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
seed = 0

model_dir = "../pretrained_models/c2img"
resolution = 256
num_stages = 4
num_inference_steps = [10, 10, 10, 10]
guidance_scale = 0.0
shift = 1.0
class_label = 10
num_examples = 4

# Input image for synthetic measurement
image_path = "../assets/example.png"  # replace with a valid path

# Inpainting operator
operator_cfg = dict(
    name="inpainting",
    mask_type="box",
    mask_len_range=(80, 160),
    mask_prob_range=None,
    image_size=resolution,
    margin=(32, 32),
    sigma=0.05,
)

# MCMC / PS hyperparams
mcmc_steps = 20
mcmc_lr = 1e-5
mcmc_tau = 0.05
mcmc_lr_min_ratio = 0.1

seed_everything(seed)
print("device:", device)

In [ ]:
# ===== Load model / scheduler =====
config = OmegaConf.load(os.path.join(model_dir, "config.yaml"))
model = config_utils.instantiate_from_config(config.model).to(device)
ckpt = torch.load(os.path.join(model_dir, "model.pt"), map_location="cpu", weights_only=False)

if isinstance(ckpt, dict) and "model" in ckpt:
    model.load_state_dict(ckpt["model"], strict=True)
else:
    model.load_state_dict(ckpt, strict=True)

model.eval()
scheduler = PixelFlowScheduler(
    num_train_timesteps=config.scheduler.num_train_timesteps,
    num_stages=num_stages,
    gamma=-1 / 3,
)

print("model loaded")

In [ ]:
# ===== Build measurement =====
transform = transforms.Compose([
    transforms.Resize((resolution, resolution)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])

img = Image.open(image_path).convert("RGB")
gt = transform(img).unsqueeze(0).to(device)
gt = gt.repeat(num_examples, 1, 1, 1)

operator = get_operator(**operator_cfg)
y = operator.measure(gt).to(device)

mcmc = MCMCSampler(
    num_steps=mcmc_steps,
    lr=mcmc_lr,
    tau=mcmc_tau,
    lr_min_ratio=mcmc_lr_min_ratio,
    prior_solver="gaussian",
    mc_algo="langevin",
)

print("measurement ready:", y.shape)

In [ ]:
def sample_block_noise(scheduler, bs, ch, height, width, eps=1e-6):
    gamma = scheduler.gamma
    cov = torch.eye(4) * (1 - gamma) + torch.ones(4, 4) * gamma + eps * torch.eye(4)
    dist = torch.distributions.multivariate_normal.MultivariateNormal(torch.zeros(4), cov)
    block_number = bs * ch * (height // 2) * (width // 2)
    noise = torch.stack([dist.sample() for _ in range(block_number)])
    noise = noise.view(bs, ch, height // 2, width // 2, 2, 2).permute(0, 1, 2, 4, 3, 5).reshape(bs, ch, height, width)
    return noise


def stage_measurement(y_full, h, w):
    return F.interpolate(y_full, size=(h, w), mode="bilinear", align_corners=False)

In [ ]:
# ===== Procedural PS sampling loop (no pipeline/class wrapper) =====
prompt_embeds = torch.tensor([class_label] * num_examples, dtype=torch.int32, device=device)
neg_prompt = 1000 * torch.ones_like(prompt_embeds)
prompt_embeds = torch.cat([neg_prompt, prompt_embeds], dim=0)

init_factor = 2 ** (num_stages - 1)
h = resolution // init_factor
w = resolution // init_factor
latents = randn_tensor((num_examples, 3, h, w), device=device, dtype=torch.float32)

for stage_idx in range(num_stages):
    scheduler.set_timesteps(num_inference_steps[stage_idx], stage_idx, device=device, shift=shift)
    Timesteps = scheduler.Timesteps

    if stage_idx > 0:
        h, w = h * 2, w * 2
        latents = F.interpolate(latents, size=(h, w), mode="nearest")
        original_start_t = scheduler.original_start_t[stage_idx]
        gamma = scheduler.gamma
        alpha = 1 / (math.sqrt(1 - (1 / gamma)) * (1 - original_start_t) + original_start_t)
        beta = alpha * (1 - original_start_t) / math.sqrt(-gamma)
        noise = sample_block_noise(scheduler, *latents.shape).to(device=device, dtype=latents.dtype)
        latents = alpha * latents + beta * noise

    size_tensor = torch.tensor([latents.shape[-1] // model.patch_size], dtype=torch.int32, device=device)
    pos_embed = get_2d_rotary_pos_embed(
        embed_dim=model.attention_head_dim,
        crops_coords=((0, 0), (latents.shape[-1] // model.patch_size, latents.shape[-1] // model.patch_size)),
        grid_size=(latents.shape[-1] // model.patch_size, latents.shape[-1] // model.patch_size),
        output_type="pt",
    )
    rope_pos = torch.stack(pos_embed, -1)

    for T in Timesteps:
        latent_model_input = torch.cat([latents, latents], dim=0)
        timestep = T.expand(latent_model_input.shape[0]).to(latent_model_input.dtype)
        error_pred = model(
            latent_model_input,
            timestep=timestep,
            class_labels=prompt_embeds,
            latent_size=size_tensor,
            pos_embed=rope_pos,
        )

        # classifier-free guidance
        eps_u, eps_c = error_pred.chunk(2)
        error_pred = eps_u + guidance_scale * (eps_c - eps_u)

        # x0 estimate from current state
        ratio = float(T.detach().item()) / 1000.0
        x0_hat = latents + (1.0 - ratio) * error_pred

        # posterior correction with MCMC
        y_stage = stage_measurement(y, x0_hat.shape[-2], x0_hat.shape[-1])
        sigma = max(1e-4, 1.0 - ratio)
        x0_hat = mcmc.sample(
            xt=latents,
            model=None,
            x0hat=x0_hat,
            operator=operator,
            measurement=y_stage,
            sigma=sigma,
            ratio=ratio,
            verbose=False,
        )

        x1_hat = torch.randn_like(x0_hat)
        latents = scheduler.step_x0_hat(x0_hat=x0_hat, x1_hat=x1_hat, stage_index=stage_idx)

samples = (latents / 2 + 0.5).clamp(0, 1).cpu().permute(0, 2, 3, 1).float().numpy()
print("done", samples.shape)

In [ ]:
# ===== Visualize =====
n = min(len(samples), num_examples)
fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
if n == 1:
    axes = [axes]

for i in range(n):
    axes[i].imshow(samples[i])
    axes[i].axis("off")
    axes[i].set_title(f"sample {i}")

plt.tight_layout()
plt.show()